# Part E: Tree-Based Regression Models
**Robust Regression Engine**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")

TARGET    = 'house_price_inr'
DROP_COLS = ['property_id', 'sale_date']
FEATURES  = [col for col in df.columns if col not in [TARGET] + DROP_COLS]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

## Task 15 — Decision Tree Regression

In [ ]:
model = DecisionTreeRegressor()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
comparison = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

comparison.head(10)

In [ ]:
print("MAE   :", mean_absolute_error(y_test, y_pred))
print("MSE   :", mean_squared_error(y_test, y_pred))
print("RMSE  :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score :", r2_score(y_test, y_pred))

In [ ]:
train_score = model.score(X_train, y_train)
test_score  = model.score(X_test, y_test)

print("Training Score :", train_score)
print("Testing Score  :", test_score)

In [ ]:
plt.figure(figsize=(25, 10))
plot_tree(model, max_depth=3, feature_names=FEATURES, filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree (Top 3 Levels)")
plt.show()

In [ ]:
# check tree depth
print("Depth of Tree :", model.get_depth())

In [ ]:
# check total nodes
print("Number of Leaf Nodes :", model.get_n_leaves())

In [ ]:
# total features used
print("Number of Features :", model.n_features_in_)

In [ ]:
# which column is important
importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': model.feature_importances_
})

importance.sort_values(by='Importance', ascending=False)

## Task 16 — Control Tree Complexity (Hyperparameters)

In [ ]:
depths = range(1, 21)
train_r2 = []
test_r2  = []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_r2.append(m.score(X_train, y_train))
    test_r2.append(m.score(X_test, y_test))

best_depth = list(depths)[int(np.argmax(test_r2))]

plt.figure(figsize=(9, 5))
plt.plot(depths, train_r2, label='Train R2', marker='o')
plt.plot(depths, test_r2, label='Test R2', marker='s')
plt.axvline(best_depth, color='red', linestyle='--', label=f'Best Depth={best_depth}')
plt.xlabel('max_depth')
plt.ylabel('R2 Score')
plt.title('Decision Tree: Depth vs Performance')
plt.legend()
plt.show()

print("Best max_depth :", best_depth)

## Task 16 — Pre-Pruning

In [ ]:
model2 = DecisionTreeRegressor(
    max_depth=best_depth,
    min_samples_split=12,
    min_samples_leaf=10,
    random_state=42
)

model2.fit(X_train, y_train)

y_pred2 = model2.predict(X_test)

print("Training Score :", model2.score(X_train, y_train))
print("Testing Score  :", model2.score(X_test, y_test))

In [ ]:
plt.figure(figsize=(25, 12))
plot_tree(
    model2,
    feature_names=FEATURES,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()

## Task 16 — Post-Pruning (Cost Complexity Pruning)

In [ ]:
model_pruned = DecisionTreeRegressor(
    ccp_alpha=500000,
    random_state=42
)

model_pruned.fit(X_train, y_train)

y_pred_pruned = model_pruned.predict(X_test)

print("Old Model Train   :", model.score(X_train, y_train))
print("Old Model Test    :", model.score(X_test, y_test))
print("Pruned Model Train:", model_pruned.score(X_train, y_train))
print("Pruned Model Test :", model_pruned.score(X_test, y_test))

In [ ]:
print("Old Tree Depth    :", model.get_depth())
print("Pruned Tree Depth :", model_pruned.get_depth())

print("Old Leaves    :", model.get_n_leaves())
print("Pruned Leaves :", model_pruned.get_n_leaves())

## Task 17 — Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
mae  = mean_absolute_error(y_test, y_pred_rf)
mse  = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred_rf)

print("MAE      :", mae)
print("MSE      :", mse)
print("RMSE     :", rmse)
print("R2 Score :", r2)

In [ ]:
# Feature Importance
importance_rf = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
})

importance_rf = importance_rf.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(importance_rf['Feature'], importance_rf['Importance'], color='steelblue')
plt.title("Random Forest Feature Importances")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Task 18 — Single Tree vs Ensemble

In [ ]:
best_dt = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_dt.fit(X_train, y_train)

print(f"{'Model':<30} {'Train R2':>10} {'Test R2':>10} {'RMSE':>15}")
print("-" * 70)

for name, m, yp in [
    ('Decision Tree (default)', model, y_pred),
    (f'Decision Tree (depth={best_depth})', best_dt, best_dt.predict(X_test)),
    ('Random Forest (100 trees)', rf, y_pred_rf),
]:
    tr   = m.score(X_train, y_train)
    te   = m.score(X_test, y_test)
    rmse = np.sqrt(mean_squared_error(y_test, yp))
    print(f"{name:<30} {tr:>10.4f} {te:>10.4f} {rmse:>15,.0f}")

## OOB Score — Random Forest

In [ ]:
rf_oob = RandomForestRegressor(
    n_estimators=100,
    oob_score=True,
    random_state=42
)

rf_oob.fit(X_train, y_train)

print("OOB Score :", rf_oob.oob_score_)